In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/3Dgs_mipnerf360",exist_ok=True)
print("done succesfully") ###for sanity check

In [ ]:
!apt-get install -y -q \
    libglew-dev libassimp-dev libboost-all-dev \
    libgtk-3-dev libopencv-dev libglfw3-dev \
    libavdevice-dev libavcodec-dev libeigen3-dev \
    libxxf86vm-dev ffmpeg

!pip install -q torch torchvision \
    --index-url https://download.pytorch.org/whl/cu118
!pip install -q plyfile tqdm lpips

In [ ]:
%cd /content
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive -q
%cd gaussian-splatting

!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

print("3DGS installed ")

In [ ]:
%cd /content/gaussian-splatting

!wget -q http://storage.googleapis.com/gresearch/refraw360/360_v2.zip
!unzip -q 360_v2.zip

!ls 360_v2/
# bicycle bonsai counter garden kitchen room stump

In [ ]:
!python train.py \
    -s /content/gaussian-splatting/bicycle \
    -m /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full \
    --eval \
    -i images_4

print("Full training done ")

In [ ]:
!python render.py \
    -m /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full \
    --skip_train  # only render test views

print("Test renders done")

In [ ]:
!python metrics.py \
    -m /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full

# Print results
import json
with open('/content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full/results.json') as f:
    r = json.load(f)

print("\n FULL 3DGS RESULTS")
for split, metrics in r.items():
    print(f"\n{split}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
import os, glob

# Explicitly grab the TRAIN renders (169 frames = smooth video)
renders_dir = '/content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full/train/ours_30000/renders'

print(f"Using renders from: {renders_dir}")
print(f"Number of frames: {len(os.listdir(renders_dir))}")

# -vf pad fixes odd-dimension error (pads to nearest even number)
!ffmpeg -y -framerate 12 \
    -pattern_type glob \
    -i '{renders_dir}/*.png' \
    -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" \
    -c:v libx264 \
    -pix_fmt yuv420p \
    -crf 18 \
    /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full_video.mp4

print("Video done ")

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_video(path):
    mp4 = open(path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width=800 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>""")

show_video('/content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full_video.mp4')
# ```

# ---



In [ ]:
import os, shutil, numpy as np

scene_src = "/content/gaussian-splatting/bicycle"
scene_dst = "/content/gaussian-splatting/bicycle_sparse"

# Copy full scene (we need the COLMAP poses)
if os.path.exists(scene_dst):
    shutil.rmtree(scene_dst)
shutil.copytree(scene_src, scene_dst)

# Replace images with only N views
src_imgs = os.path.join(scene_src, "images_4")
dst_imgs = os.path.join(scene_dst, "images_4")
shutil.rmtree(dst_imgs)
os.makedirs(dst_imgs)

all_images  = sorted(os.listdir(src_imgs))
train_imgs  = [img for i, img in enumerate(all_images) if i % 8 != 0]

N = 6  #  change to 8 or 12 to experiment
indices  = np.linspace(0, len(train_imgs)-1, N, dtype=int)
selected = [train_imgs[i] for i in indices]

for img in selected:
    shutil.copy(os.path.join(src_imgs, img),
                os.path.join(dst_imgs, img))

print(f"Total images:    {len(all_images)}")
print(f"Train images:    {len(train_imgs)}")
print(f"Selected {N} views: {selected}")

In [ ]:
import os, shutil, numpy as np

scene_src = "/content/gaussian-splatting/bicycle"
scene_dst = "/content/gaussian-splatting/bicycle_sparse"

# Fresh copy of entire scene including ALL images
if os.path.exists(scene_dst):
    shutil.rmtree(scene_dst)
shutil.copytree(scene_src, scene_dst)

# Get train image list (exclude every 8th = test)
src_imgs = os.path.join(scene_src, "images_4")
all_images = sorted(os.listdir(src_imgs))
train_imgs = [img for i, img in enumerate(all_images) if i % 8 != 0]

# Select N evenly spaced
N = 6
indices = np.linspace(0, len(train_imgs)-1, N, dtype=int)
selected = set(train_imgs[i] for i in indices)
test_imgs = set(all_images) - set(train_imgs)  # keep test images too

# Remove non-selected TRAIN images only
# Keep: selected train images + all test images
dst_imgs = os.path.join(scene_dst, "images_4")
for img in os.listdir(dst_imgs):
    if img not in selected and img not in test_imgs:
        os.remove(os.path.join(dst_imgs, img))

remaining = os.listdir(dst_imgs)
print(f"Total original images:  {len(all_images)}")
print(f"Train images:           {len(train_imgs)}")
print(f"Selected sparse views:  {len(selected)}")
print(f"Test images kept:       {len(test_imgs)}")
print(f"Images in sparse folder:{len(remaining)}")
print(f"\nSelected: {sorted(selected)}")

In [ ]:
# Read the file
with open('/content/gaussian-splatting/utils/camera_utils.py', 'r') as f:
    content = f.read()

print(content)  # see what's there first

In [ ]:
old_code = '''def cameraList_from_camInfos(cam_infos, resolution_scale, args, is_nerf_synthetic, is_test_dataset):
    camera_list = []

    for id, c in enumerate(cam_infos):
        camera_list.append(loadCam(args, id, c, resolution_scale, is_nerf_synthetic, is_test_dataset))

    return camera_list'''

new_code = '''def cameraList_from_camInfos(cam_infos, resolution_scale, args, is_nerf_synthetic, is_test_dataset):
    camera_list = []

    for id, c in enumerate(cam_infos):
        # Skip missing images (for sparse training)
        if not os.path.exists(c.image_path):
            continue
        camera_list.append(loadCam(args, id, c, resolution_scale, is_nerf_synthetic, is_test_dataset))

    return camera_list'''

with open('/content/gaussian-splatting/utils/camera_utils.py', 'r') as f:
    content = f.read()

# Add os import if not present
if 'import os' not in content:
    content = 'import os\n' + content

content = content.replace(old_code, new_code)

with open('/content/gaussian-splatting/utils/camera_utils.py', 'w') as f:
    f.write(content)

print("Patched ")

In [ ]:
!python train.py \
    -s /content/gaussian-splatting/bicycle_sparse \
    -m /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_sparse_6 \
    --eval \
    -i images_4

print("Sparse training done ✓")

In [ ]:
!python render.py \
    -m /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_sparse_6 \
    --skip_train

!python metrics.py \
    -m /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_sparse_6

import json
with open('/content/drive/MyDrive/3DGS_MipNeRF360/bicycle_sparse_6/results.json') as f:
    r = json.load(f)

print("\n SPARSE 3DGS RESULTS")
for split, metrics in r.items():
    print(f"\n{split}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
import os


full_test   = '/content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full/test/ours_30000/renders'
sparse_test = '/content/drive/MyDrive/3DGS_MipNeRF360/bicycle_sparse_6/test/ours_30000/renders'

print(f"Full test frames:   {len(os.listdir(full_test))}")
print(f"Sparse test frames: {len(os.listdir(sparse_test))}")

In [ ]:
# Full model test video
!ffmpeg -y -framerate 5 \
    -pattern_type glob \
    -i '{full_test}/*.png' \
    -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" \
    -c:v libx264 -pix_fmt yuv420p -crf 18 \
    /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full_test_video.mp4

# Sparse model test video
!ffmpeg -y -framerate 5 \
    -pattern_type glob \
    -i '{sparse_test}/*.png' \
    -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" \
    -c:v libx264 -pix_fmt yuv420p -crf 18 \
    /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_sparse_6_test_video.mp4

print("Both test videos done ")

In [ ]:
# Side-by-side comparison with labels
!ffmpeg -y \
    -i /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_full_test_video.mp4 \
    -i /content/drive/MyDrive/3DGS_MipNeRF360/bicycle_sparse_6_test_video.mp4 \
    -filter_complex "\
        [0:v]scale=-2:540,pad=ceil(iw/2)*2:ceil(ih/2)*2,\
        drawtext=text='Full 3DGS (194 views)':fontsize=22:fontcolor=white:x=10:y=10,\
        drawtext=text='PSNR 25.24 | SSIM 0.766':fontsize=18:fontcolor=yellow:x=10:y=40[left];\
        [1:v]scale=-2:540,pad=ceil(iw/2)*2:ceil(ih/2)*2,\
        drawtext=text='Sparse 3DGS (6 views)':fontsize=22:fontcolor=white:x=10:y=10,\
        drawtext=text='PSNR 15.74 | SSIM 0.248':fontsize=18:fontcolor=red:x=10:y=40[right];\
        [left][right]hstack=inputs=2[out]" \
    -map "[out]" \
    -c:v libx264 -pix_fmt yuv420p -crf 18 \
    /content/drive/MyDrive/3DGS_MipNeRF360/comparison_test_video.mp4

print("Comparison video done ")

In [ ]:
# Preview
from IPython.display import HTML
from base64 import b64encode

def show_video(path, label=""):
    mp4 = open(path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <h3>{label}</h3>
    <video width=900 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>""")

display(show_video(
    '/content/drive/MyDrive/3DGS_MipNeRF360/comparison_test_video.mp4',
    'Test Set Comparison — Left: Full 3DGS | Right: Sparse 3DGS'
))